# 13 — Does local neighbourhood information change the main conclusions?

The preceding analyses treat each location as an independent row. This notebook adds a simple graph model so that a location can also learn from nearby locations. The purpose is not to search for a new winning architecture. It is to test whether the main representation findings remain recognisable when local spatial relationships are built into the model.

Eight representative specifications are evaluated with the same five held-out borough groups used in the main analysis. A feature-only neural network is fitted alongside GATv2 with the same inputs and training rule. Their difference therefore gives a direct, limited indication of what neighbourhood message passing adds beyond the neural network itself.


## Models included

The comparison is deliberately small and covers the central dissertation contrasts.

The initial 250-epoch ceiling was insufficient for two rich-control EPC graph models. A 500-epoch refinement resolved those cases, but three PTAL-baseline MLP folds selected epochs 499, 500 and 500. The final specification therefore uses a 1,000-epoch ceiling and treats values within the final 2% as boundary selections. The graph, architecture, validation and all other training settings remain unchanged.

**PTAL**

- location controls only;
- location controls + DINOv2;
- location controls + all representations.

**EPC**

- compact property controls only;
- compact controls + DINOv2;
- compact controls + all representations;
- richer property controls only;
- richer controls + TESSERA.

For every specification, two heads are fitted: a multilayer perceptron (MLP), which uses only the location's own features, and GATv2, which also exchanges information with nearby locations. Ridge remains the primary transparent benchmark; this notebook is a selected robustness analysis.


In [ ]:
# Connect Google Drive and install the graph-learning package if needed.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import subprocess
import warnings
warnings.filterwarnings('ignore')

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric==2.6.1'
])

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric

from torch_geometric.nn import GATv2Conv
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import radius_neighbors_graph
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path('/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE')
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({name: getattr(_config, name) for name in dir(_config) if not name.startswith('__')})

pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 240)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.set_float32_matmul_precision('high')

print('Python:', platform.python_version())
print('numpy:', np.__version__, 'pandas:', pd.__version__, 'sklearn:', sklearn.__version__)
print('torch:', torch.__version__, 'torch-geometric:', torch_geometric.__version__)
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    print('A T4 or L4 GPU runtime is strongly recommended before the full run.')


## 1. Load the frozen sample, features and borough folds

The graph analysis uses the same observations, outcomes, feature definitions and held-out borough groups as the preceding models. File fingerprints are checked before training so a graph result cannot be compared with a different version of the data.


In [ ]:
# Verify the completed prerequisite chain and load the canonical model table.
required_paths = [
    FINAL_MODEL_TABLE_PATH,
    FEATURE_MANIFEST_JSON_PATH,
    RIDGE_OUTER_FOLDS_PATH,
    INCREMENTAL_RUN_SPEC_PATH,
    INCREMENTAL_AUDIT_PATH,
    INCREMENTAL_RESULTS_PATH,
    XGBOOST_AUDIT_PATH,
    XGBOOST_SUMMARY_PATH,
    GATV2_V1_RESULTS_PATH,
    GATV2_V1_AUDIT_PATH,
    GATV2_V2_RESULTS_PATH,
    GATV2_V2_AUDIT_PATH,
]
for path in required_paths:
    assert path.exists(), f'Missing prerequisite: {path}'

for audit_path in [INCREMENTAL_AUDIT_PATH, XGBOOST_AUDIT_PATH]:
    audit = json.loads(audit_path.read_text())
    assert audit['integrity_gate_pass'] is True
    assert audit['interpretation_gate_pass'] is True

df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
manifest = json.loads(FEATURE_MANIFEST_JSON_PATH.read_text())
source_07_spec = json.loads(INCREMENTAL_RUN_SPEC_PATH.read_text())
source_07_audit = json.loads(INCREMENTAL_AUDIT_PATH.read_text())
source_13_v1_audit = json.loads(GATV2_V1_AUDIT_PATH.read_text())
source_13_v2_audit = json.loads(GATV2_V2_AUDIT_PATH.read_text())
source_13_v2_results = pd.read_csv(GATV2_V2_RESULTS_PATH)

assert source_13_v1_audit['integrity_gate_pass'] is True
assert source_13_v1_audit['interpretation_gate_pass'] is False
assert source_13_v1_audit['maximum_epochs'] == 250
assert source_13_v1_audit['models_with_repeated_max_epoch_hits'] == 2
assert source_13_v2_audit['integrity_gate_pass'] is True
assert source_13_v2_audit['interpretation_gate_pass'] is True
assert source_13_v2_audit['maximum_epochs'] == 500
v2_near = source_13_v2_results[
    (source_13_v2_results['task'] == 'PTAL')
    & (source_13_v2_results['model_id'] == 'PTAL_spatial_baseline')
    & (source_13_v2_results['head'] == 'MLP')
]
assert len(v2_near) == 5
assert int((v2_near['best_epoch'] >= 490).sum()) == 3

target_col = manifest['target_column']
group_col = manifest['group_column']
categorical_master = set(manifest['categorical_columns'])
feature_sets = manifest['feature_sets']

assert len(df) == 26597
assert df['sample_id'].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert {'x_km', 'y_km'}.issubset(df.columns)
assert df['task'].value_counts().to_dict() == {'EPC': 20000, 'PTAL': 6597}

df = df.sort_values(['task', 'sample_id'], kind='mergesort').reset_index(drop=True)
task_counts = df['task'].value_counts().to_dict()

model_key_frame = df[['sample_id', 'task', group_col, target_col]].copy()
model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(model_key_frame, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()

assert source_07_spec['model_key_sha256'] == model_key_hash
assert source_07_spec['feature_manifest_sha256'] == manifest_hash
assert source_07_audit['model_key_sha256'] == model_key_hash
assert source_07_audit['feature_manifest_sha256'] == manifest_hash

print('Frozen prerequisite chain: PASS')
print('Samples:', task_counts)


In [ ]:
# Reconstruct the eight pre-selected feature specifications.
SELECTED_MODEL_IDS = [
    'PTAL_spatial_baseline',
    'PTAL_spatial_baseline__plus__DINOv2',
    'PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata',
    'EPC_controls_sparse',
    'EPC_controls_sparse__plus__DINOv2',
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata',
    'EPC_controls_extensive',
    'EPC_controls_extensive__plus__TESSERA',
]

def dedupe(columns):
    return list(dict.fromkeys(columns))

model_features = {
    'PTAL_spatial_baseline': list(feature_sets['PTAL_spatial_baseline']),
    'PTAL_spatial_baseline__plus__DINOv2': dedupe(feature_sets['PTAL_spatial_baseline'] + feature_sets['DINOv2']),
    'PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata': dedupe(feature_sets['PTAL_spatial_baseline'] + feature_sets['All_representations_plus_SV_metadata']),
    'EPC_controls_sparse': list(feature_sets['EPC_controls_sparse']),
    'EPC_controls_sparse__plus__DINOv2': dedupe(feature_sets['EPC_controls_sparse'] + feature_sets['DINOv2']),
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata': dedupe(feature_sets['EPC_controls_sparse'] + feature_sets['All_representations_plus_SV_metadata']),
    'EPC_controls_extensive': list(feature_sets['EPC_controls_extensive']),
    'EPC_controls_extensive__plus__TESSERA': dedupe(feature_sets['EPC_controls_extensive'] + feature_sets['TESSERA']),
}

source_specs = pd.DataFrame(source_07_spec['model_specifications']).set_index('model_id')
assert set(SELECTED_MODEL_IDS).issubset(source_specs.index)

def columns_sha256(columns):
    return hashlib.sha256(json.dumps(list(columns), separators=(',', ':')).encode()).hexdigest()

def optional_text(value):
    return None if pd.isna(value) else str(value)

model_records = []
for model_id in SELECTED_MODEL_IDS:
    source = source_specs.loc[model_id]
    cols = model_features[model_id]
    assert cols and not [c for c in cols if c not in df.columns]
    assert len(cols) == int(source['n_features_manifest'])
    assert columns_sha256(cols) == source['feature_columns_sha256']
    model_records.append({
        'task': source['task'],
        'model_id': model_id,
        'baseline_id': optional_text(source['baseline_id']),
        'control_family': source['control_family'],
        'analysis_role': source['analysis_role'],
        'added_feature_set': optional_text(source['added_feature_set']),
        'n_features_manifest': len(cols),
        'feature_columns_sha256': columns_sha256(cols),
    })

model_specs = pd.DataFrame(model_records)
assert model_specs.groupby('task').size().to_dict() == {'EPC': 5, 'PTAL': 3}
display(model_specs)


## 2. Prepare the inputs without using held-out information

Missing values, category encoding and feature scaling are learned from the relevant training subset only. Street View structural absence is handled in the same way as the main benchmark. The outcome is standardised using the training subset and converted back to its original units for reporting.


In [ ]:
# Preserve the established Street View and fold-local preprocessing rules.
SV_META_COLS = ['sv_has_streetview', 'sv_n_images', 'sv_min_dist_m', 'sv_mean_dist_m']
SV_CLIP_COLS = feature_sets['StreetView_CLIP_only']

def apply_structural_sv_metadata_fill(task_df, task):
    out = task_df.copy()
    radius = {'PTAL': float(STREETVIEW_PTAL_RADIUS_M), 'EPC': float(STREETVIEW_EPC_RADIUS_M)}[task]
    has = pd.to_numeric(out['sv_has_streetview'], errors='coerce')
    clip_complete = out[SV_CLIP_COLS].notna().all(axis=1)
    clip_all_missing = out[SV_CLIP_COLS].isna().all(axis=1)
    has = has.where(has.notna(), clip_complete.astype(int)).astype(int)
    assert has.isin([0, 1]).all()
    assert clip_complete[has.eq(1)].all()
    assert clip_all_missing[has.eq(0)].all()
    out['sv_has_streetview'] = has
    for col in SV_META_COLS[1:]:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    no_sv = has.eq(0)
    out.loc[no_sv, 'sv_n_images'] = 0.0
    for col in ['sv_min_dist_m', 'sv_mean_dist_m']:
        out.loc[no_sv & out[col].isna(), col] = radius
    return out

def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False, dtype=np.float32)

def build_preprocessor(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []
    if numeric_cols:
        transformers.append((
            'num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]), numeric_cols
        ))
    if categorical_cols:
        transformers.append((
            'cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
                ('onehot', make_onehot()),
            ]), categorical_cols
        ))
    return ColumnTransformer(transformers, remainder='drop', sparse_threshold=0.0)

def to_float32(matrix):
    if hasattr(matrix, 'toarray'):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapped = series.astype(str).str.strip().str.lower().map({
        'true': True, 'false': False, '1': True, '0': False,
    })
    assert mapped.notna().all(), f'Unrecognised Boolean values: {series[mapped.isna()].unique()}'
    return mapped.astype(bool)


## 3. Build local graphs without crossing data partitions

Two locations are connected when they lie within a fixed local radius: 1 km for the spatially thinned PTAL sample and 500 m for the denser postcode sample. The graph uses coordinates only, never the outcome.

Training, early-validation and held-out test graphs are built separately. There are therefore no train-to-test or train-to-validation edges. At prediction time, a held-out location may use the observed features of other held-out locations in its local area, but no held-out outcome is available to the model. This represents inductive prediction for a previously unseen part of London whose input layers are available.


In [ ]:
# Construct an undirected radius graph and retain basic graph diagnostics.
GRAPH_RADIUS_KM = {'PTAL': 1.0, 'EPC': 0.5}

def build_graph(coords_km, radius_km):
    coords = np.asarray(coords_km, dtype=np.float32)
    assert coords.ndim == 2 and coords.shape[1] == 2 and np.isfinite(coords).all()
    adjacency = radius_neighbors_graph(
        coords, radius=float(radius_km), mode='connectivity',
        include_self=False, n_jobs=-1,
    )
    adjacency = adjacency.maximum(adjacency.T).tocoo()
    edge_index_np = np.vstack([adjacency.row, adjacency.col]).astype(np.int64)

    degree = np.bincount(edge_index_np[0], minlength=len(coords))
    if edge_index_np.shape[1]:
        delta = coords[edge_index_np[0]] - coords[edge_index_np[1]]
        distances = np.sqrt((delta ** 2).sum(axis=1))
        max_distance = float(distances.max())
        median_distance = float(np.median(distances))
        assert max_distance <= float(radius_km) + 1e-5
    else:
        max_distance = 0.0
        median_distance = 0.0

    stats = {
        'n_nodes': int(len(coords)),
        'n_directed_edges': int(edge_index_np.shape[1]),
        'mean_degree': float(degree.mean()),
        'median_degree': float(np.median(degree)),
        'isolated_share': float((degree == 0).mean()),
        'median_edge_distance_km': median_distance,
        'max_edge_distance_km': max_distance,
    }
    return torch.as_tensor(edge_index_np, dtype=torch.long), stats

for task in ['PTAL', 'EPC']:
    task_df = df[df['task'] == task]
    _, qa = build_graph(task_df[['x_km', 'y_km']].to_numpy(), GRAPH_RADIUS_KM[task])
    print(task, qa)


## 4. Fix the neural-network training rule

Both heads use the same 64-unit input projection and a 32-unit prediction layer. GATv2 inserts two attention layers between them; the MLP replaces those layers with ordinary fully connected layers. Architecture and optimisation settings are fixed before any held-out result is inspected.

Within each outer training set, complete boroughs are reserved for early stopping. The chosen epoch count is then used to train a fresh model on the complete outer training set before the held-out boroughs are predicted.


In [ ]:
# Define matched feature-only and graph-aware neural heads.
HIDDEN_DIM = 64
GAT_HEADS = 4
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 1000
NEAR_BOUNDARY_SHARE = 0.98
PATIENCE = 25
MIN_EPOCHS = 20
EARLY_VALIDATION_SHARE = 0.20

class FeatureMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.input = nn.Linear(in_dim, HIDDEN_DIM)
        self.hidden = nn.Linear(HIDDEN_DIM, 32)
        self.output = nn.Linear(32, 1)

    def forward(self, x, edge_index=None):
        x = F.elu(self.input(x))
        x = F.dropout(x, p=DROPOUT, training=self.training)
        x = F.elu(self.hidden(x))
        x = F.dropout(x, p=DROPOUT, training=self.training)
        return self.output(x).squeeze(-1)

class LocalGATv2(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.input = nn.Linear(in_dim, HIDDEN_DIM)
        self.gat1 = GATv2Conv(
            HIDDEN_DIM, HIDDEN_DIM // GAT_HEADS,
            heads=GAT_HEADS, concat=True, dropout=DROPOUT,
            add_self_loops=True, share_weights=False,
        )
        self.gat2 = GATv2Conv(
            HIDDEN_DIM, 32, heads=1, concat=False,
            dropout=DROPOUT, add_self_loops=True, share_weights=False,
        )
        self.output = nn.Linear(32, 1)

    def forward(self, x, edge_index):
        x = F.elu(self.input(x))
        x = F.dropout(x, p=DROPOUT, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=DROPOUT, training=self.training)
        x = F.elu(self.gat2(x, edge_index))
        return self.output(x).squeeze(-1)

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_model(head, in_dim):
    if head == 'MLP':
        return FeatureMLP(in_dim)
    if head == 'GATv2':
        return LocalGATv2(in_dim)
    raise ValueError(head)

@torch.no_grad()
def predict_standardised(model, x, edge_index):
    model.eval()
    return model(x, edge_index).detach().cpu().numpy()

def choose_epoch(head, X_train, y_train, edge_train, X_valid, y_valid, edge_valid, seed):
    set_seed(seed)
    y_mean = float(np.mean(y_train))
    y_sd = float(np.std(y_train))
    assert np.isfinite(y_sd) and y_sd > 0

    xt = torch.as_tensor(X_train, dtype=torch.float32, device=DEVICE)
    xv = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor((y_train - y_mean) / y_sd, dtype=torch.float32, device=DEVICE)
    et = edge_train.to(DEVICE)
    ev = edge_valid.to(DEVICE)

    model = make_model(head, X_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_epoch = 1
    best_rmse = np.inf
    waiting = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pred = model(xt, et)
        loss = F.mse_loss(pred, yt)
        assert torch.isfinite(loss)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        pred_valid = predict_standardised(model, xv, ev) * y_sd + y_mean
        valid_rmse = rmse(y_valid, pred_valid)
        if valid_rmse < best_rmse - 1e-5:
            best_rmse = valid_rmse
            best_epoch = epoch
            waiting = 0
        else:
            waiting += 1
        if epoch >= MIN_EPOCHS and waiting >= PATIENCE:
            break

    del model, optimizer, xt, xv, yt, et, ev
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return int(best_epoch), float(best_rmse)

def fit_fixed_epochs(head, X_train, y_train, edge_train, X_test, edge_test, epochs, seed):
    set_seed(seed)
    y_mean = float(np.mean(y_train))
    y_sd = float(np.std(y_train))
    assert np.isfinite(y_sd) and y_sd > 0

    xt = torch.as_tensor(X_train, dtype=torch.float32, device=DEVICE)
    xs = torch.as_tensor(X_test, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor((y_train - y_mean) / y_sd, dtype=torch.float32, device=DEVICE)
    et = edge_train.to(DEVICE)
    es = edge_test.to(DEVICE)

    model = make_model(head, X_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    for _ in range(int(epochs)):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pred = model(xt, et)
        loss = F.mse_loss(pred, yt)
        assert torch.isfinite(loss)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

    pred_test = predict_standardised(model, xs, es) * y_sd + y_mean
    del model, optimizer, xt, xs, yt, et, es
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pred_test.astype(np.float32, copy=False)


In [ ]:
# Reconstruct and verify the exact five borough-held-out assignments.
outer_assignment_rows = []
fold_index_by_task = {}

for task in ['PTAL', 'EPC']:
    task_df = df[df['task'] == task].reset_index(drop=True)
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = np.full(len(task_df), -1, dtype=int)
    splitter = GroupKFold(n_splits=int(source_07_spec['outer_splits']))

    for fold, (_, test_idx) in enumerate(splitter.split(task_df, task_df[target_col], groups)):
        task_fold[test_idx] = fold
        for idx in test_idx:
            outer_assignment_rows.append({
                'sample_id': task_df.loc[idx, 'sample_id'],
                'task': task,
                'outer_fold': int(fold),
                'borough_code': task_df.loc[idx, group_col],
                'target': float(task_df.loc[idx, target_col]),
            })

    assert (task_fold >= 0).all()
    fold_index_by_task[task] = task_fold

outer_folds = (
    pd.DataFrame(outer_assignment_rows)
    .sort_values(['task', 'sample_id'], kind='mergesort')
    .reset_index(drop=True)
)
saved_outer_folds = (
    pd.read_csv(RIDGE_OUTER_FOLDS_PATH)
    .sort_values(['task', 'sample_id'], kind='mergesort')
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(
    saved_outer_folds[outer_folds.columns], outer_folds,
    check_dtype=False, check_exact=False, rtol=0, atol=1e-12,
)

fold_assignment_hash = hashlib.sha256(
    pd.util.hash_pandas_object(outer_folds, index=False).values.tobytes()
).hexdigest()
assert source_07_spec['outer_fold_assignment_sha256'] == fold_assignment_hash
print('Verified frozen borough-fold SHA256:', fold_assignment_hash)


In [ ]:
# Freeze the complete graph-model specification before fitting any held-out fold.
run_spec = {
    'run_spec_version': '13-gatv2-v3-2026-08-24',
    'notebook': '13_gatv2_neighbourhood_robustness.ipynb',
    'analysis_role': 'selected graph-neighbourhood robustness; not primary model selection',
    'model_key_sha256': model_key_hash,
    'feature_manifest_sha256': manifest_hash,
    'outer_fold_assignment_sha256': fold_assignment_hash,
    'source_07_run_spec_sha256': source_07_audit['run_spec_sha256'],
    'source_13_v1_run_spec_sha256': source_13_v1_audit['run_spec_sha256'],
    'source_13_v2_run_spec_sha256': source_13_v2_audit['run_spec_sha256'],
    'refinement_history': 'maximum epochs 250 to 500 to 1000; all other graph, model, validation and optimisation settings unchanged',
    'near_epoch_boundary_share': NEAR_BOUNDARY_SHARE,
    'device_type': DEVICE.type,
    'torch_version': torch.__version__,
    'torch_geometric_version': torch_geometric.__version__,
    'selected_models': model_records,
    'heads': ['MLP', 'GATv2'],
    'outer_splits': 5,
    'graph_radius_km': GRAPH_RADIUS_KM,
    'graph_rule': 'undirected radius graph built separately within train, validation and test partitions; self loops added by GATv2Conv',
    'early_validation_grouping': 'one deterministic GroupShuffleSplit of outer-training boroughs',
    'early_validation_share': EARLY_VALIDATION_SHARE,
    'refit_rule': 'best epoch from group-held validation, then fresh refit on full outer training data',
    'training': {
        'hidden_dim': HIDDEN_DIM,
        'gat_heads': GAT_HEADS,
        'dropout': DROPOUT,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'maximum_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'minimum_epochs': MIN_EPOCHS,
        'loss': 'mean squared error on outer-training-standardised target',
        'gradient_clip_norm': 5.0,
    },
    'preprocessing': {
        'numeric_imputation': 'training-subset median',
        'numeric_scaling': 'training-subset StandardScaler',
        'categorical_imputation': 'training-subset constant __MISSING__',
        'categorical_encoding': 'training-subset OneHotEncoder(handle_unknown=ignore)',
        'streetview_structural_absence': 'same rules as Notebook 07',
    },
    'random_state': int(RANDOM_STATE),
}
run_spec_sha256 = hashlib.sha256(json.dumps(run_spec, sort_keys=True).encode()).hexdigest()

if GATV2_RUN_SPEC_PATH.exists():
    existing = json.loads(GATV2_RUN_SPEC_PATH.read_text())
    assert existing == run_spec, (
        'Existing Notebook-13 checkpoints belong to a different specification or runtime device. '
        'Do not mix them; archive the existing 13 outputs before a documented rerun.'
    )
else:
    atomic_json(run_spec, GATV2_RUN_SPEC_PATH)

expected_keys = {
    (row['task'], row['model_id'], head, fold)
    for row in model_records for head in ['MLP', 'GATv2'] for fold in range(5)
}
expected_prediction_rows = int(
    3 * task_counts['PTAL'] * 2 + 5 * task_counts['EPC'] * 2
)
assert len(expected_keys) == 80
assert expected_prediction_rows == 239582

print('Notebook-13 run-spec SHA256:', run_spec_sha256)
print('Expected neural fold-runs:', len(expected_keys))
print('Expected prediction rows:', expected_prediction_rows)


## 5. Fit the feature-only and graph-aware heads

Each completed prediction is saved independently. If Colab disconnects, rerunning the notebook validates finished files and resumes from the first missing task, model, head and fold. The held-out borough outcomes are used only for the final reported score.


In [ ]:
# Check that an existing prediction file belongs to the exact intended run.
def checkpoint_is_valid(path, task, model_id, head, outer_fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            'sample_id', 'task', 'model_id', 'head', 'outer_fold',
            'borough_code', 'y_true', 'y_pred', 'run_spec_sha256',
        }
        if not required.issubset(p.columns) or p['sample_id'].duplicated().any():
            return False
        if not p['task'].eq(task).all() or not p['model_id'].eq(model_id).all():
            return False
        if not p['head'].eq(head).all() or not p['outer_fold'].astype(int).eq(outer_fold).all():
            return False
        if not p['run_spec_sha256'].eq(run_spec_sha256).all():
            return False
        if not np.isfinite(p['y_true']).all() or not np.isfinite(p['y_pred']).all():
            return False
        return set(p['sample_id'].astype(str)) == set(pd.Series(expected_ids).astype(str))
    except Exception:
        return False

if GATV2_RESULTS_PATH.exists():
    completed = pd.read_csv(GATV2_RESULTS_PATH)
    assert not completed.duplicated(['task', 'model_id', 'head', 'outer_fold']).any()
    assert completed['run_spec_sha256'].eq(run_spec_sha256).all()
    result_rows = completed.to_dict('records')
else:
    result_rows = []


In [ ]:
# Run all eight specifications under the frozen borough folds.
for task in ['PTAL', 'EPC']:
    task_df = df[df['task'] == task].reset_index(drop=True)
    task_df = apply_structural_sv_metadata_fill(task_df, task)
    y = pd.to_numeric(task_df[target_col], errors='raise').to_numpy(dtype=np.float32)
    groups = task_df[group_col].astype(str).to_numpy()
    coords = task_df[['x_km', 'y_km']].to_numpy(dtype=np.float32)
    task_fold = fold_index_by_task[task]
    radius_km = GRAPH_RADIUS_KM[task]

    for spec in [r for r in model_records if r['task'] == task]:
        model_id = spec['model_id']
        cols = model_features[model_id]
        X = task_df[cols]

        for outer_fold in range(5):
            test_idx = np.flatnonzero(task_fold == outer_fold)
            train_idx = np.flatnonzero(task_fold != outer_fold)
            expected_ids = task_df.iloc[test_idx]['sample_id'].to_numpy()
            complete_heads = []
            existing_keys = {
                (r['task'], r['model_id'], r['head'], int(r['outer_fold']))
                for r in result_rows
            }
            for head in ['MLP', 'GATv2']:
                key = (task, model_id, head, outer_fold)
                path = GATV2_CHUNK_DIR / f'{task}__{model_id}__{head}__fold{outer_fold}.parquet'
                if key in existing_keys and checkpoint_is_valid(
                    path, task, model_id, head, outer_fold, expected_ids
                ):
                    complete_heads.append(head)
            if set(complete_heads) == {'MLP', 'GATv2'}:
                print('SKIP validated pair:', task, model_id, outer_fold)
                continue

            print('\n' + '=' * 100)
            print(task, '|', model_id, '| held-out borough fold', outer_fold)
            print('=' * 100)

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            g_train = groups[train_idx]

            inner_splitter = GroupShuffleSplit(
                n_splits=1, test_size=EARLY_VALIDATION_SHARE,
                random_state=RANDOM_STATE + outer_fold,
            )
            inner_train, inner_valid = next(inner_splitter.split(X_train, y_train, g_train))
            assert set(g_train[inner_train]).isdisjoint(set(g_train[inner_valid]))

            # Early-stopping matrices and graphs use the inner-training fit only.
            pre_early = build_preprocessor(cols)
            Xe_train = to_float32(pre_early.fit_transform(X_train.iloc[inner_train]))
            Xe_valid = to_float32(pre_early.transform(X_train.iloc[inner_valid]))
            edge_early_train, qa_early_train = build_graph(coords[train_idx][inner_train], radius_km)
            edge_early_valid, qa_early_valid = build_graph(coords[train_idx][inner_valid], radius_km)

            # Final matrices and graphs use every outer-training row but remain separate from test.
            pre_full = build_preprocessor(cols)
            Xf_train = to_float32(pre_full.fit_transform(X_train))
            Xf_test = to_float32(pre_full.transform(X_test))
            edge_full_train, qa_full_train = build_graph(coords[train_idx], radius_km)
            edge_test, qa_test = build_graph(coords[test_idx], radius_km)

            for head in ['MLP', 'GATv2']:
                run_key = (task, model_id, head, outer_fold)
                pred_file = GATV2_CHUNK_DIR / f'{task}__{model_id}__{head}__fold{outer_fold}.parquet'
                existing_keys = {
                    (r['task'], r['model_id'], r['head'], int(r['outer_fold']))
                    for r in result_rows
                }
                if run_key in existing_keys and checkpoint_is_valid(
                    pred_file, task, model_id, head, outer_fold, expected_ids
                ):
                    print('SKIP validated checkpoint:', run_key)
                    continue

                seed = RANDOM_STATE + outer_fold + (1000 if head == 'GATv2' else 0)
                t0 = time.time()
                best_epoch, best_valid_rmse = choose_epoch(
                    head,
                    Xe_train, y_train[inner_train], edge_early_train,
                    Xe_valid, y_train[inner_valid], edge_early_valid,
                    seed,
                )
                pred = fit_fixed_epochs(
                    head, Xf_train, y_train, edge_full_train,
                    Xf_test, edge_test, best_epoch, seed,
                )
                elapsed_s = time.time() - t0
                assert len(pred) == len(test_idx) and np.isfinite(pred).all()

                row = {
                    **spec,
                    'head': head,
                    'outer_fold': int(outer_fold),
                    'n_train': int(len(train_idx)),
                    'n_test': int(len(test_idx)),
                    'n_train_boroughs': int(len(np.unique(g_train))),
                    'n_test_boroughs': int(len(np.unique(groups[test_idx]))),
                    'n_early_train': int(len(inner_train)),
                    'n_early_valid': int(len(inner_valid)),
                    'transformed_feature_count': int(Xf_train.shape[1]),
                    'best_epoch': int(best_epoch),
                    'max_epoch_boundary': bool(best_epoch == MAX_EPOCHS),
                    'near_epoch_boundary': bool(best_epoch >= int(np.ceil(NEAR_BOUNDARY_SHARE * MAX_EPOCHS))),
                    'early_valid_rmse': float(best_valid_rmse),
                    'r2': float(r2_score(y_test, pred)),
                    'rmse': rmse(y_test, pred),
                    'mae': float(mean_absolute_error(y_test, pred)),
                    'fit_seconds': float(elapsed_s),
                    'device': DEVICE.type,
                    'graph_radius_km': float(radius_km),
                    'train_mean_degree': qa_full_train['mean_degree'],
                    'train_isolated_share': qa_full_train['isolated_share'],
                    'test_mean_degree': qa_test['mean_degree'],
                    'test_isolated_share': qa_test['isolated_share'],
                    'test_max_edge_distance_km': qa_test['max_edge_distance_km'],
                    'run_spec_sha256': run_spec_sha256,
                }

                pred_frame = pd.DataFrame({
                    'sample_id': task_df.iloc[test_idx]['sample_id'].to_numpy(),
                    'task': task,
                    'model_id': model_id,
                    'head': head,
                    'outer_fold': int(outer_fold),
                    'borough_code': groups[test_idx],
                    'y_true': y_test,
                    'y_pred': pred,
                    'run_spec_sha256': run_spec_sha256,
                })
                atomic_parquet(pred_frame, pred_file)

                result_rows = [
                    r for r in result_rows
                    if (r['task'], r['model_id'], r['head'], int(r['outer_fold'])) != run_key
                ]
                result_rows.append(row)
                results_now = pd.DataFrame(result_rows).sort_values(
                    ['task', 'model_id', 'head', 'outer_fold'], kind='mergesort'
                )
                atomic_csv(results_now, GATV2_RESULTS_PATH)
                print(head, '| epoch', best_epoch, '| R2', f'{row["r2"]:.4f}', '| minutes', f'{elapsed_s / 60:.2f}')

            del pre_early, pre_full, Xe_train, Xe_valid, Xf_train, Xf_test
            del edge_early_train, edge_early_valid, edge_full_train, edge_test
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print('Neural fitting complete or safely checkpointed.')


## 6. Verify every held-out prediction

Final summaries are created only after all 80 task–model–head–fold runs and all 239,582 held-out predictions match the intended sample IDs. Partial output remains resumable but is never labelled as a complete result.


In [ ]:
# Assemble and validate the complete prediction set.
results = pd.read_csv(GATV2_RESULTS_PATH)
results['outer_fold'] = results['outer_fold'].astype(int)
results['max_epoch_boundary'] = coerce_bool(results['max_epoch_boundary'])
results['near_epoch_boundary'] = coerce_bool(results['near_epoch_boundary'])
actual_keys = set(map(tuple, results[['task', 'model_id', 'head', 'outer_fold']].to_numpy()))
missing_keys = sorted(expected_keys - actual_keys)
unexpected_keys = sorted(actual_keys - expected_keys)
print('Completed fold-runs:', len(actual_keys), '/', len(expected_keys))
print('Missing:', len(missing_keys), 'Unexpected:', len(unexpected_keys))
assert not missing_keys, 'Notebook 13 is incomplete: rerun the fitting section.'
assert not unexpected_keys
assert results['run_spec_sha256'].eq(run_spec_sha256).all()

pred_frames = []
for task, model_id, head, outer_fold in sorted(expected_keys):
    path = GATV2_CHUNK_DIR / f'{task}__{model_id}__{head}__fold{outer_fold}.parquet'
    task_df = df[df['task'] == task].reset_index(drop=True)
    task_fold = fold_index_by_task[task]
    expected_ids = task_df.loc[task_fold == outer_fold, 'sample_id'].to_numpy()
    assert checkpoint_is_valid(path, task, model_id, head, outer_fold, expected_ids)
    pred_frames.append(pd.read_parquet(path))

preds = pd.concat(pred_frames, ignore_index=True)
assert len(preds) == expected_prediction_rows
assert not preds.duplicated(['sample_id', 'task', 'model_id', 'head', 'outer_fold']).any()
atomic_parquet(preds, GATV2_PREDICTIONS_PATH)

print('Validated prediction chunks:', len(pred_frames))
print('Validated prediction rows:', len(preds))


## 7. Separate the graph effect from the representation effect

Three summaries are produced. The first reports absolute performance. The second compares GATv2 with the matched feature-only MLP on the same held-out boroughs. The third compares each representation-enhanced model with its own control baseline separately for MLP and GATv2. This prevents a change in neural architecture from being mistaken for the contribution of an embedding.


In [ ]:
# Summarise absolute out-of-fold performance for every head and feature specification.
summary_rows = []
for (task, model_id, head), fold_group in results.groupby(['task', 'model_id', 'head']):
    pred_group = preds[
        (preds['task'] == task) &
        (preds['model_id'] == model_id) &
        (preds['head'] == head)
    ]
    assert len(fold_group) == 5 and len(pred_group) == task_counts[task]
    summary_rows.append({
        'task': task,
        'model_id': model_id,
        'head': head,
        'n_features': int(fold_group['n_features_manifest'].iloc[0]),
        'mean_r2': float(fold_group['r2'].mean()),
        'sd_r2': float(fold_group['r2'].std(ddof=1)),
        'mean_rmse': float(fold_group['rmse'].mean()),
        'sd_rmse': float(fold_group['rmse'].std(ddof=1)),
        'mean_mae': float(fold_group['mae'].mean()),
        'sd_mae': float(fold_group['mae'].std(ddof=1)),
        'pooled_r2': float(r2_score(pred_group['y_true'], pred_group['y_pred'])),
        'pooled_rmse': rmse(pred_group['y_true'], pred_group['y_pred']),
        'pooled_mae': float(mean_absolute_error(pred_group['y_true'], pred_group['y_pred'])),
        'median_best_epoch': float(fold_group['best_epoch'].median()),
        'max_epoch_boundary_hits': int(fold_group['max_epoch_boundary'].sum()),
        'near_epoch_boundary_hits': int(fold_group['near_epoch_boundary'].sum()),
        'mean_test_degree': float(fold_group['test_mean_degree'].mean()),
        'mean_test_isolated_share': float(fold_group['test_isolated_share'].mean()),
        'total_fit_minutes': float(fold_group['fit_seconds'].sum() / 60),
    })

gat_summary = pd.DataFrame(summary_rows).sort_values(
    ['task', 'model_id', 'head'], kind='mergesort'
)
atomic_csv(gat_summary, GATV2_SUMMARY_PATH)
display(gat_summary)


In [ ]:
# Pair GATv2 and MLP results within each held-out borough fold.
mlp = results[results['head'] == 'MLP'][
    ['task', 'model_id', 'outer_fold', 'r2', 'rmse', 'mae']
].copy()
gat = results[results['head'] == 'GATv2'][
    ['task', 'model_id', 'outer_fold', 'r2', 'rmse', 'mae']
].copy()

head_pairs = gat.merge(
    mlp, on=['task', 'model_id', 'outer_fold'],
    how='inner', validate='one_to_one', suffixes=('_gatv2', '_mlp')
)
assert len(head_pairs) == 40
head_pairs['delta_r2_gatv2_minus_mlp'] = head_pairs['r2_gatv2'] - head_pairs['r2_mlp']
head_pairs['delta_rmse_gatv2_minus_mlp'] = head_pairs['rmse_gatv2'] - head_pairs['rmse_mlp']
head_pairs['delta_mae_gatv2_minus_mlp'] = head_pairs['mae_gatv2'] - head_pairs['mae_mlp']
head_pairs['r2_win'] = head_pairs['delta_r2_gatv2_minus_mlp'] > 0
head_pairs['rmse_win'] = head_pairs['delta_rmse_gatv2_minus_mlp'] < 0
head_pairs['mae_win'] = head_pairs['delta_mae_gatv2_minus_mlp'] < 0
atomic_csv(head_pairs, GATV2_VS_MLP_FOLD_PATH)

head_summary_rows = []
for (task, model_id), g in head_pairs.groupby(['task', 'model_id']):
    head_summary_rows.append({
        'task': task,
        'model_id': model_id,
        'mean_delta_r2_gatv2_minus_mlp': float(g['delta_r2_gatv2_minus_mlp'].mean()),
        'sd_delta_r2': float(g['delta_r2_gatv2_minus_mlp'].std(ddof=1)),
        'r2_wins_out_of_5': int(g['r2_win'].sum()),
        'mean_delta_rmse_gatv2_minus_mlp': float(g['delta_rmse_gatv2_minus_mlp'].mean()),
        'rmse_wins_out_of_5': int(g['rmse_win'].sum()),
        'mean_delta_mae_gatv2_minus_mlp': float(g['delta_mae_gatv2_minus_mlp'].mean()),
        'mae_wins_out_of_5': int(g['mae_win'].sum()),
    })
gat_vs_mlp_summary = pd.DataFrame(head_summary_rows)
atomic_csv(gat_vs_mlp_summary, GATV2_VS_MLP_SUMMARY_PATH)
display(gat_vs_mlp_summary.sort_values(['task', 'mean_delta_r2_gatv2_minus_mlp'], ascending=[True, False]))


In [ ]:
# Measure representation increments within each neural head.
BASELINE_MAP = {
    'PTAL_spatial_baseline__plus__DINOv2': 'PTAL_spatial_baseline',
    'PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata': 'PTAL_spatial_baseline',
    'EPC_controls_sparse__plus__DINOv2': 'EPC_controls_sparse',
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata': 'EPC_controls_sparse',
    'EPC_controls_extensive__plus__TESSERA': 'EPC_controls_extensive',
}

increment_rows = []
for head in ['MLP', 'GATv2']:
    head_results = results[results['head'] == head]
    for model_id, baseline_id in BASELINE_MAP.items():
        model_part = head_results[head_results['model_id'] == model_id]
        base_part = head_results[head_results['model_id'] == baseline_id]
        assert len(model_part) == 5 and len(base_part) == 5
        paired = model_part.merge(
            base_part, on=['task', 'outer_fold'],
            how='inner', validate='one_to_one', suffixes=('_model', '_baseline')
        )
        for _, r in paired.iterrows():
            increment_rows.append({
                'task': r['task'], 'head': head,
                'model_id': model_id, 'baseline_id': baseline_id,
                'outer_fold': int(r['outer_fold']),
                'delta_r2': float(r['r2_model'] - r['r2_baseline']),
                'delta_rmse': float(r['rmse_model'] - r['rmse_baseline']),
                'delta_mae': float(r['mae_model'] - r['mae_baseline']),
            })

incremental = pd.DataFrame(increment_rows)
assert len(incremental) == 50
assert not incremental.duplicated(['task', 'head', 'model_id', 'outer_fold']).any()
atomic_csv(incremental, GATV2_INCREMENTAL_FOLD_PATH)

incremental_summary = (
    incremental
    .assign(
        r2_win=lambda x: x['delta_r2'] > 0,
        rmse_win=lambda x: x['delta_rmse'] < 0,
        mae_win=lambda x: x['delta_mae'] < 0,
    )
    .groupby(['task', 'head', 'model_id', 'baseline_id'], as_index=False)
    .agg(
        mean_delta_r2=('delta_r2', 'mean'),
        sd_delta_r2=('delta_r2', 'std'),
        r2_wins_out_of_5=('r2_win', 'sum'),
        mean_delta_rmse=('delta_rmse', 'mean'),
        rmse_wins_out_of_5=('rmse_win', 'sum'),
        mean_delta_mae=('delta_mae', 'mean'),
        mae_wins_out_of_5=('mae_win', 'sum'),
    )
)
assert len(incremental_summary) == 10
atomic_csv(incremental_summary, GATV2_INCREMENTAL_SUMMARY_PATH)
display(incremental_summary)


In [ ]:
# Place the neural results beside the existing Ridge and XGBoost references.
ridge = pd.read_csv(INCREMENTAL_RESULTS_PATH)
ridge = ridge[ridge['model_id'].isin(SELECTED_MODEL_IDS)]
ridge_reference = (
    ridge.groupby(['task', 'model_id'], as_index=False)
    .agg(ridge_mean_r2=('r2', 'mean'), ridge_mean_rmse=('rmse', 'mean'), ridge_mean_mae=('mae', 'mean'))
)
xgb = pd.read_csv(XGBOOST_SUMMARY_PATH)
xgb_reference = xgb[xgb['model_id'].isin(SELECTED_MODEL_IDS)][
    ['task', 'model_id', 'mean_r2', 'mean_rmse', 'mean_mae']
].rename(columns={
    'mean_r2': 'xgboost_mean_r2',
    'mean_rmse': 'xgboost_mean_rmse',
    'mean_mae': 'xgboost_mean_mae',
})
neural_reference = gat_summary.pivot(
    index=['task', 'model_id'], columns='head', values=['mean_r2', 'mean_rmse', 'mean_mae']
)
neural_reference.columns = [f'{head.lower()}_{metric}' for metric, head in neural_reference.columns]
neural_reference = neural_reference.reset_index()

reference = (
    neural_reference
    .merge(ridge_reference, on=['task', 'model_id'], how='left', validate='one_to_one')
    .merge(xgb_reference, on=['task', 'model_id'], how='left', validate='one_to_one')
)
assert len(reference) == 8 and reference.notna().all().all()
atomic_csv(reference, GATV2_REFERENCE_SUMMARY_PATH)
display(reference)


## 8. Complete the analysis and define its interpretation

The integrity check confirms that every intended run and prediction is present, every graph respects its radius, and every comparison uses matching held-out borough folds. Interpretation pauses if any task–model–head combination selects an epoch in the final 2% of the 1,000-epoch range in at least three folds, because that would suggest that the training range remains too short.


In [ ]:
# Write the final audit record only after all outputs reconcile.
max_boundary = results.groupby(['task', 'model_id', 'head'])['max_epoch_boundary'].sum()
near_boundary = results.groupby(['task', 'model_id', 'head'])['near_epoch_boundary'].sum()
repeated_boundary_models = near_boundary[near_boundary >= 3]

integrity_gate_pass = bool(
    len(actual_keys) == len(expected_keys)
    and len(preds) == expected_prediction_rows
    and len(head_pairs) == 40
    and len(incremental) == 50
    and len(reference) == 8
    and results[['r2', 'rmse', 'mae', 'best_epoch']].apply(np.isfinite).all().all()
    and (results['test_max_edge_distance_km'] <= results['graph_radius_km'] + 1e-5).all()
)
interpretation_gate_pass = bool(integrity_gate_pass and len(repeated_boundary_models) == 0)

audit_summary = {
    'run_spec_path': str(GATV2_RUN_SPEC_PATH),
    'run_spec_sha256': run_spec_sha256,
    'model_key_sha256': model_key_hash,
    'feature_manifest_sha256': manifest_hash,
    'outer_fold_assignment_sha256': fold_assignment_hash,
    'device': DEVICE.type,
    'selected_feature_specifications': 8,
    'heads_per_specification': 2,
    'expected_fold_runs': 80,
    'completed_fold_runs': int(len(actual_keys)),
    'expected_prediction_rows': int(expected_prediction_rows),
    'actual_prediction_rows': int(len(preds)),
    'validated_prediction_chunks': int(len(pred_frames)),
    'gatv2_vs_mlp_fold_pairs': int(len(head_pairs)),
    'within_head_incremental_fold_pairs': int(len(incremental)),
    'reference_comparison_models': int(len(reference)),
    'maximum_epochs': int(MAX_EPOCHS),
    'near_epoch_boundary_share': float(NEAR_BOUNDARY_SHARE),
    'models_with_repeated_exact_max_epoch_hits': int((max_boundary >= 3).sum()),
    'models_with_repeated_near_epoch_hits': int(len(repeated_boundary_models)),
    'graph_radius_km': GRAPH_RADIUS_KM,
    'maximum_test_isolated_share': float(results['test_isolated_share'].max()),
    'primary_validation': 'same frozen five borough-grouped outer folds as Notebook 07',
    'edge_separation': 'train, validation and test graphs built separately; no cross-partition edges',
    'analysis_role': 'selected graph-neighbourhood robustness; Ridge remains the primary linear probe',
    'integrity_gate_pass': integrity_gate_pass,
    'interpretation_gate_pass': interpretation_gate_pass,
}
assert integrity_gate_pass
atomic_json(audit_summary, GATV2_AUDIT_PATH)

print(json.dumps(audit_summary, indent=2))
if not interpretation_gate_pass:
    print('PAUSE INTERPRETATION: extend only the epoch ceiling and rerun affected checkpoints.')
else:
    print('Notebook 13 complete: graph-model results are ready for post-run review.')


## Main result

The final 1,000-epoch run completed all 80 model–fold fits and 239,582 held-out predictions. No task–model–head combination repeatedly selected an epoch near the training boundary, so the earlier convergence concern is resolved.

Local message passing did not generally improve the matched feature-only neural network. GATv2 was weaker for every representation-enhanced PTAL model and every selected EPC representation model. The only positive average difference occurred for the PTAL location-only baseline: GATv2 increased mean R² by 0.025, with higher R² in three of five held-out borough folds and lower MAE in four. This is a small, mixed exception rather than evidence of a general graph-model advantage.

The representation findings were more stable. Relative to compact controls, DINOv2 increased mean R² by 0.227 for the PTAL MLP and 0.185 for the EPC MLP, improving all five folds in both cases. Full fusion increased mean R² by 0.293 for the PTAL MLP and 0.218 for the compact-control EPC MLP, again improving all five folds. With richer EPC controls, TESSERA added only 0.015 mean R² to the MLP and reduced GATv2 performance.

These results support a limited robustness conclusion: the learned representations retain predictive value under neural models, but simple neighbourhood aggregation usually does not add information beyond using each location's own features directly. Ridge remains the primary transparent comparison, while GATv2 provides evidence that the main task-dependent representation conclusions are not dependent on omitting a local graph structure.
